# core

> `core` contains the `CloakDF` class — a pseudonymisation/anonymisation engine for consistent cross-table key replacement and column masking, with Fernet-encrypted storage of mappings.

In [ ]:
#| default_exp core

I've done some prior exploration here's what I propose. 

**Project: `cloakdf`** — a pseudonymisation/anonymisation package built with nbdev (fast.ai style). The agreed design:

- **One class** (e.g. `Anonymiser`) with shared state across tables
- **`key_maps`**: `{group_id: {original_val: pseudonym}}` — for consistent cross-table key pseudonymisation
- **`vault`**: stores originals of masked columns so they can be restored
- **Config per table** using key groups like `id1`, `id2`, etc.:
  ```python
  tables = {
      'patients': {'id1': 'mid',       'mask': ['name', 'dob']},
      'visits':   {'id1': 'mid',       'id2': 'visit_code'},
      'doctors':  {'id2': 'doc_id',    'mask': ['name']},
  }
  ```
- **Lookup table approach** (random UUIDs, stored in a dict) — simple and reversible

The nbdev project is already scaffolded, and we're in `00_core.ipynb` which currently just has `def foo(): pass`. 

## Setup & data

In [ ]:
#| eval: false
import httpx
from pathlib import Path

base = "https://raw.githubusercontent.com/neo4j-contrib/northwind-neo4j/refs/heads/master/data"
files = ["customers.csv", "orders.csv", "employees.csv"]
dest = Path("../data")
dest.mkdir(exist_ok=True)

for f in files:
    (dest/f).write_bytes(httpx.get(f"{base}/{f}").content)
    print(f"✓ {f}")

✓ customers.csv


✓ orders.csv
✓ employees.csv


In [ ]:
#| eval: false
import pandas as pd
for f in files:
    df = pd.read_csv(dest/f,on_bad_lines='skip' )
    print(f"\n{f}: {df.shape}")
    print(df.columns.tolist())
    display(df.head(2))


customers.csv: (67, 11)
['customerID', 'companyName', 'contactName', 'contactTitle', 'address', 'city', 'region', 'postalCode', 'country', 'phone', 'fax']


,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
0,ALFKI,Alfreds Futterkiste,Maria Anders,Sales Representative,Obere Str. 57,Berlin,NaN,12209,Germany,030-0074321,030-0076545
1,ANATR,Ana Trujillo Emparedados y helados,Ana Trujillo,Owner,Avda. de la Constitución 2222,México D.F.,NaN,05021,Mexico,(5) 555-4729,(5) 555-3745



orders.csv: (654, 14)
['orderID', 'customerID', 'employeeID', 'orderDate', 'requiredDate', 'shippedDate', 'shipVia', 'freight', 'shipName', 'shipAddress', 'shipCity', 'shipRegion', 'shipPostalCode', 'shipCountry']


,orderID,customerID,employeeID,orderDate,requiredDate,shippedDate,shipVia,freight,shipName,shipAddress,shipCity,shipRegion,shipPostalCode,shipCountry
0,10248,VINET,5,1996-07-04 00:00:00.000,1996-08-01 00:00:00.000,1996-07-16 00:00:00.000,3,32.38,Vins et alcools Chevalier,59 rue de l'Abbaye,Reims,NaN,51100,France
1,10249,TOMSP,6,1996-07-05 00:00:00.000,1996-08-16 00:00:00.000,1996-07-10 00:00:00.000,1,11.61,Toms Spezialitäten,Luisenstr. 48,Münster,NaN,44087,Germany



employees.csv: (9, 18)
['employeeID', 'lastName', 'firstName', 'title', 'titleOfCourtesy', 'birthDate', 'hireDate', 'address', 'city', 'region', 'postalCode', 'country', 'homePhone', 'extension', 'photo', 'notes', 'reportsTo', 'photoPath']


,employeeID,lastName,firstName,title,titleOfCourtesy,birthDate,hireDate,address,city,region,postalCode,country,homePhone,extension,photo,notes,reportsTo,photoPath
0,1,Davolio,Nancy,Sales Representative,Ms.,1948-12-08 00:00:00.000,1992-05-01 00:00:00.000,507 - 20th Ave. E. Apt. 2A,Seattle,WA,98122,USA,(206) 555-9857,5467,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,Education includes a BA in psychology from Col...,2.0,http://accweb/emmployees/davolio.bmp
1,2,Fuller,Andrew,"Vice President, Sales",Dr.,1952-02-19 00:00:00.000,1992-08-14 00:00:00.000,908 W. Capital Way,Tacoma,WA,98401,USA,(206) 555-9482,3457,0x151C2F00020000000D000E0014002100FFFFFFFF4269...,Andrew received his BTS commercial in 1974 and...,NaN,http://accweb/emmployees/fuller.bmp


## Table configuration

In [ ]:
#| eval: false
tables = {
    'customers': {
        'id1': 'customerID',
        'mask': ['contactName', 'address', 'phone', 'companyName', 'fax', 'city']
    },
    'orders': {
        'id1': 'customerID',
        'id2': 'employeeID',
        'mask': ['shipName', 'shipAddress']
    },
    'employees': {
        'id2': 'employeeID',
        'mask': ['firstName', 'lastName', 'address', 'homePhone', 'birthDate', 'notes']
    },
}

Okay can we sketch this out? I started with these data structures and the North Wind dataset #Should I start building the Unknown class; We need English UK 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great config — that maps nicely to the design from the start notebook. A few thoughts on the sketch:

**Class name:** `Unknown` (as you suggested) — and we'll use UK English throughout (`anonymise`, `pseudonymise`, etc.)

**Core state:**
- `tables` — the config you've already written
- `key_maps` — `{group: {orig_val: pseudonym}}` e.g. `{'id1': {'ALFKI': 'uuid-xxx', ...}}`
- `vault` — stores original masked column values per table, so you can restore them

**Key methods to start with:**
1. `pseudonymise(name, df)` — takes a table name + DataFrame, pseudonymises keys and masks columns, returns the cleaned DataFrame
2. `restore(name, df)` — reverses the process

Does that match your thinking? If so, go ahead and stub out the class with `__init__` — just setting up those three attributes from the config. I'd suggest starting simple: take `tables` as the only `__init__` parameter.

## Building the `CloakDF` class

In [ ]:
#| eval: false
class CloakDF(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df): pass
    def decode(self, name,df): pass

#Simple start. Let's review.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice clean stub! A few notes:

- **`encode`/`decode`** — good naming choice, cleaner than `pseudonymise`/`restore`
- `(object)` — you can drop that, it's implicit in Python 3, but no harm keeping it
- You might want `#| export` above the class so nbdev exports it to your module

For `encode`, the logic will be roughly:
1. **Pseudonymise keys** — for each `id*` group in `tables[name]`, map original values → UUIDs (reusing existing mappings from `key_maps`)
2. **Vault & mask** — for each column in `mask`, stash the originals in `vault[name]`, then blank/redact them

Want to tackle step 1 first — the key pseudonymisation bit inside `encode`?

## Key pseudonymisation

In [ ]:
#| eval: false
for t in tables.keys():
    print("table is->", t)
    print(tables[t])

table is-> customers
{'id1': 'customerID', 'mask': ['contactName', 'address', 'phone', 'companyName', 'fax', 'city']}
table is-> orders
{'id1': 'customerID', 'id2': 'employeeID', 'mask': ['shipName', 'shipAddress']}
table is-> employees
{'id2': 'employeeID', 'mask': ['firstName', 'lastName', 'address', 'homePhone', 'birthDate', 'notes']}


In [ ]:
#| eval: false
ids = {k: v for k, v in tables['orders'].items() if k.startswith('id')}
ids

{'id1': 'customerID', 'id2': 'employeeID'}

In [ ]:
#| eval: false
customers = pd.read_csv(dest/"customers.csv", on_bad_lines='skip')
customers.sample(2)

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
24,ISLAT,Island Trading,Helen Bennett,Marketing Manager,Garden House Crowther Way,Cowes,Isle of Wight,PO31 7PJ,UK,(198) 555-8888,NaN
45,QUICK,QUICK-Stop,Horst Kloss,Accounting Manager,Taucherstraße 10,Cunewalde,NaN,01307,Germany,0372-035188,NaN


In [ ]:
#| eval: false
unique_customers = customers['customerID'].unique()
unique_customers
import uuid
u = uuid.uuid4()

OK, I got as far as this. Before I continue, explain briefly:

1. why uuid4 is this a more secure id format
2. i assume i can from each uuid transform this into hex, int or other format? that would be the id itself

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good questions!

**1. Why uuid4?** It's generated from **cryptographically random** bits — there's no timestamp, MAC address, or sequence number baked in. That means you can't reverse-engineer any info about *when* or *where* it was created. With 122 random bits, the chance of a collision is astronomically small (you'd need to generate billions per second for decades). That's what makes it ideal for pseudonymisation — the fake ID reveals nothing about the original.

**2. Formats** — Yes, exactly. A UUID is just a 128-bit number. You can represent it as:
- `str(u)` → `'a3b2c1d4-...'` (the familiar hyphenated hex)
- `u.hex` → `'a3b2c1d4...'` (32 hex chars, no hyphens)
- `u.int` → a big integer
- `u.bytes` → 16 raw bytes

For your key maps, `str(u)` is probably the most readable choice. You could also use `u.hex` if you want something shorter and cleaner.

Ready to build that `{original: uuid}` mapping dict for the customerID column?

In [ ]:
#| eval: false
{i:str(uuid.uuid4()) for i in unique_customers[:3]}

{'ALFKI': 'b26f730e-dcb2-4555-8260-1bf1a14c6980',
 'ANATR': '4c3215d4-7d7d-41f0-ab84-88e6b6ee0847',
 'ANTON': '13970cbb-c38a-48ec-86d7-cea6df50dc88'}

OK, let's rewrite this into the class. 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice — that comprehension is exactly the core logic. Now think about where it fits in `encode`:

1. Extract the id groups (you already know how: `k.startswith('id')`)
2. For each group, check if `self.key_maps[group]` already exists — if so, reuse it; if not, build it from the unique values
3. Replace the column values using the map

A hint for step 2–3: `df[col].map(mapping)` will swap values using a dict.

Have a go at writing the key pseudonymisation part of `encode` — just that bit, not the masking yet.

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        # extract the ids
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            if group not in self.key_maps: self.key_maps[group] = {}
            uniques = df[col].dropna().unique()
            for u in uniques:
                if u not in self.key_maps[group]: self.key_maps[group][u] = str(uuid.uuid4())
    def decode(self, name,df): pass

I'm not super happy with my code so far, let's review if I can make this more readable

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Your logic is solid — it's just a bit dense. Two quick wins:

1. **`setdefault`** replaces the `if group not in` check:
   ```python
   km = self.key_maps.setdefault(group, {})
   ```
   This returns the existing dict or creates an empty one — one line instead of two.

2. **Inner loop** — once you have `km`, you can tighten it:
   ```python
   for val in df[col].dropna().unique():
       km.setdefault(val, str(uuid.uuid4()))
   ```
   Same pattern — only inserts if the key is missing.

That removes both `if` statements and makes the intent clearer. Give those two changes a try and see how it reads.

I've never used setdefault; could you explain this a bit more? Is this a recent Python addition?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠

Not recent at all — it's been in Python since version 2.0! It's just one of those dict methods that many people overlook.

Here's what it does:

```python
d = {}
d.setdefault('x', 42)   # 'x' not in d, so it sets d['x'] = 42 and returns 42
d.setdefault('x', 99)   # 'x' already in d, so it does nothing and returns 42
```

So it's a **get-or-insert** in one call. It always returns the value — either the existing one or the newly inserted default. That's what makes it perfect for building up dicts of dicts:

```python
km = self.key_maps.setdefault(group, {})
```

After this line, `km` is a reference to the inner dict — whether it already existed or was just created. So any changes you make to `km` are reflected in `self.key_maps[group]` too.

Does that make sense? Try it out in a quick test cell if you want to see it in action.

In [ ]:
#| eval: false
d = {}
d.setdefault('x', 42)   # 'x' not in d, so it sets d['x'] = 42 and returns 42
d.setdefault('x', 99)   # 'x' already in d, so it does nothing and returns 42

42

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        df =df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            km = self.key_maps.setdefault(group, {}) # will create or retreive; sourcery :) 
            for val in  df[col].dropna().unique(): km.setdefault(val, str(uuid.uuid4()))
            df[col] = df[col].map(km)
        return df 
    def decode(self, name,df): pass


Let's review this code iteration.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Looking much cleaner! Key improvements:

- **`df.copy()`** — smart addition, avoids mutating the original DataFrame
- **`setdefault`** — reads much more clearly now
- **`df[col].map(km)`** — does the actual swap, nice and concise
- **Returns `df`** — so you get the pseudonymised version back

One thing to be aware of: `map(km)` will turn any value **not** in the dict into `NaN` — including values that were already `NaN`. That's probably fine for ID columns, but worth knowing.

Want to test it on the customers table and see what comes out?

In [ ]:
#| eval: false
unk = Unknown(tables)
unk.encode('customers', customers).sample(2)

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
14,8a72ff87-5752-4ee9-b49d-d7ac492da170,Ernst Handel,Roland Mendel,Sales Manager,Kirchgasse 6,Graz,NaN,8010,Austria,7675-3425,7675-3426
53,e5df65f4-018c-45ad-8b49-d47325e9fbee,Simons bistro,Jytte Petersen,Owner,Vinbæltet 34,Kobenhavn,NaN,1734,Denmark,31 12 34 56,31 13 35 57


OK, this seems to work. Let's add pseudonymise and mask methods and call it when encoding. Please review

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good refactor idea — separate concerns. Here's the shape I'd suggest:

**`_pseudonymise(self, name, df)`** — extracts the key-mapping logic you already have in `encode` (the `ids` loop). Returns the modified `df`.

**`_mask(self, name, df)`** — stores the original values of the `mask` columns in `self.vault[name]`, then replaces them with something (e.g. `'***'` or `None`). Returns the modified `df`.

**`encode`** becomes just:
```python
def encode(self, name, df):
    df = df.copy()
    df = self._pseudonymise(name, df)
    df = self._mask(name, df)
    return df
```

I'd use underscore-prefixed names (`_pseudonymise`, `_mask`) since they're internal — the public API is just `encode`/`decode`.

For `_mask`, the key question is: how do you want to store originals in the vault? One natural choice is `self.vault[name] = df[mask_cols].copy()` — stash the whole sub-DataFrame before blanking.

Have a go at writing those two methods!

## Column masking

In [ ]:
#| eval: false
class CloakDF(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}
    def encode(self, name,df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)
 
    def _pseudonymise_keys(self, name,df):
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            km = self.key_maps.setdefault(group, {})
            for val in  df[col].dropna().unique(): km.setdefault(val, str(uuid.uuid4()))
            df[col] = df[col].map(km)
        return df
    
    def _mask_columns(self, name, df): pass     
    
    
    def decode(self, name,df): pass

In [ ]:
#| eval: false
unk = CloakDF(tables)

unk._pseudonymise_keys('customers', customers).sample(10)

,customerID,companyName,contactName,contactTitle,address,city,region,postalCode,country,phone,fax
29,41200b77-a59b-4386-b042-b82c76006d51,Lehmanns Marktstand,Renate Messner,Sales Representative,Magazinweg 7,Frankfurt a.M.,NaN,60528,Germany,069-0245984,069-0245874
20,a52e20f0-2d65-41c8-adc3-889cdca9d2d2,GROSELLA-Restaurante,Manuel Pereira,Owner,5ª Ave. Los Palos Grandes,Caracas,DF,1081,Venezuela,(2) 283-2951,(2) 283-3397
41,5bdc6314-61ab-4359-a9b5-eead90768510,Ottilies Käseladen,Henriette Pfalzheim,Owner,Mehrheimerstr. 369,Köln,NaN,50739,Germany,0221-0644327,0221-0765721
17,ad31018a-2d01-4cbf-8002-4cb366067773,Franchi S.p.A.,Paolo Accorti,Sales Representative,Via Monte Bianco 34,Torino,NaN,10100,Italy,011-4988260,011-4988261
60,d0360ade-71a6-4316-91fb-0ddf596f3a05,Vaffeljernet,Palle Ibsen,Sales Manager,Smagsloget 45,Århus,NaN,8200,Denmark,86 21 32 43,86 22 33 44
52,2cb349ca-060a-4fa2-9296-b0e0886ab1cb,Seven Seas Imports,Hari Kumar,Sales Manager,90 Wadhurst Rd.,London,NaN,OX15 4NB,UK,(171) 555-1717,(171) 555-5646
57,bf8d5052-25e4-4c87-a30f-7b94e0465f05,Toms Spezialitäten,Karin Josephs,Marketing Manager,Luisenstr. 48,Münster,NaN,44087,Germany,0251-031259,0251-035695
66,be7a6942-28e7-4a8e-adbb-e79a44f0446e,Wolski Zajazd,Zbyszek Piestrzeniewicz,Owner,ul. Filtrowa 68,Warszawa,NaN,01-012,Poland,(26) 642-7012,(26) 642-7012
23,3d8ab5f0-68c7-4517-8089-89d7c186bf2d,Hungry Owl All-Night Grocers,Patricia McKenna,Sales Associate,8 Johnstown Road,Cork,Co. Cork,NaN,Ireland,2967 542,2967 3333
15,e03b9939-6b86-432f-a201-035a2aad9ecd,Folk och fä HB,Maria Larsson,Owner,Åkergatan 24,Bräcke,NaN,S-844 67,Sweden,0695-34 67 21,NaN


Let's use a hex truncated key for the mask and store in a dict across all dataframes. This way same names or addresses can be retrieved. Also let's allow nulls here not a problem to decode. I will address these in ids

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking — a shared vault means if the same address appears in both `customers` and `orders`, it gets the same pseudonym. So `vault` becomes a flat lookup like `key_maps`:

- `self.vault = {}` — `{original_value: truncated_hex}` shared across all tables
- NaNs pass through untouched
- Something like `uuid.uuid4().hex[:12]` for the replacement (you pick the length)

The logic in `_mask_columns` would be very similar to `_pseudonymise_keys` — loop over the `mask` columns, build mappings in `self.vault`, then `df[col].map(self.vault)`. The key difference is you'd want to skip NaNs rather than mapping them.

Hint: `df[col].map(self.vault)` already leaves NaN as NaN, so that's handled for free.

Have a go at implementing `_mask_columns`?

In [ ]:
#| eval: false
class Unknown(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}

    def encode(self, name, df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def _remap(self, df, col, mapping, id_fn):
        for val in df[col].dropna().unique():
            mapping.setdefault(val, id_fn())
        df[col] = df[col].map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def decode(self, name, df): pass

OK, review code for now after latest changes; highlight these.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Nice iteration! Here's what changed and why it's good:

**`_remap` helper** — You extracted the common pattern (loop uniques → setdefault → map) into one method, with `id_fn` as the generator. Both `_pseudonymise_keys` and `_mask_columns` now use it. That's a solid DRY refactor.

**`assert` on key columns** — Smart. IDs shouldn't be null — catching that early avoids silent bugs downstream. Good call separating this concern from `_mask_columns` which is fine with nulls.

**Shared `self.vault`** — Flat dict across all tables, so `"Maria Anders"` gets the same hex whether it appears in customers or elsewhere. Exactly what you described.

**`hex[:8]`** — 8 hex chars = 32 bits ≈ 4 billion possibilities. Should be fine for this dataset, but worth considering if you'd want longer for production use.

One thing to watch: `_remap` is called from both methods, but `_pseudonymise_keys` does the `df.copy()` while `_mask_columns` doesn't — that's fine since `_pseudonymise_keys` always runs first in `encode`, but it's an implicit dependency. Something to keep in mind.

Want to test it end-to-end now?

In [ ]:
#| eval: false
unk = CloakDF(tables)
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    print(f"\n--- {name} ---")
    display(unk.encode(name, df).head(3))


--- customers ---


AttributeError: 'NoneType' object has no attribute 'head'

In [ ]:
#| eval: false
unk.key_maps

{'id1': {'ALFKI': '12d452aa-e4db-4973-9593-3857840a5d83',
  'ANATR': '2780256a-de90-48fc-81cc-403803200fb1',
  'ANTON': '67f3c637-cccc-4c66-aae2-228f1854a37e',
  'AROUT': '3c52a886-011b-4716-9514-87ac5270c31b',
  'BERGS': 'd202c9ce-ccb9-4476-88ce-75f5cf137a5e',
  'BLAUS': 'a3edaf78-599e-4891-85f4-fef57c4dab04',
  'BOTTM': '214f7977-fb52-4432-941d-913ddd952362',
  'BSBEV': 'e2134e3e-2d8c-4401-8be6-f8f6bc0d6941',
  'CACTU': '26998351-85bb-435f-a730-276809877877',
  'CENTC': 'dd3e0a71-afa7-44b5-bd09-fcaba0e068d2',
  'CHOPS': '5b12531c-0bce-4f5f-a758-2ae9577d8ba2',
  'CONSH': 'e43326ca-d55b-431e-808f-f14061fdc27b',
  'DRACD': 'e2b58ee5-aaf2-4e23-a8bd-6165868580a9',
  'EASTC': 'e2dfd20f-fc54-41ff-b18f-59bd5331f1d1',
  'ERNSH': 'b0417a40-a3e1-49ee-8859-33c604b3e3bc',
  'FOLKO': '3eb690ee-eda7-4105-afa5-277a06c40bee',
  'FRANK': '3492c9a0-a3ff-4a4d-bcc6-76c543e4c20c',
  'FRANS': '81bdce4e-f0ba-4d51-bec5-0ba018d42bda',
  'FURIB': '16b076dd-cf57-45b7-837f-a91c849e49a2',
  'GREAL': 'c55ff1ed-f3b

In [ ]:
#| eval: false
unk.vault

{}

Let's work on decoding this now, the inverse property. Requirement: I also would like something to store the encoded values. Now let's just not assume that these would be available within the class because likely this could be used at the end of a pipeline 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good thinking — in a real pipeline, encoding and decoding would likely happen in separate steps (maybe even separate machines). So you need two things:

1. **Persist the mappings** — `key_maps` and `vault` need to be saveable/loadable (JSON is natural since they're just nested dicts of strings)
2. **`decode`** — invert the mappings (`{pseudonym: original}`) and `.map()` back

A couple of design questions for you:

- For storage, are you happy with a simple `save(path)` / `load(path)` pair using JSON? Or do you want something fancier?
- Should `decode` work from the in-memory state (if available) **and** from a loaded file? Or only from file?

For a start, simple save and restore is fine. But given this is sensitive data, I would like to discuss some encryption at rest.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good instinct — those mappings are essentially the keys to de-anonymise everything, so they're the most sensitive artefact in the whole pipeline.

For encryption at rest, the simplest solid option is **Fernet** from the `cryptography` library:
- Symmetric encryption (one key encrypts and decrypts)
- AES-128-CBC under the hood, with HMAC authentication
- Password → key derivation via PBKDF2 is built in

The flow would be:
1. **Save**: JSON serialize → encrypt with Fernet → write bytes to file
2. **Load**: read bytes → decrypt with Fernet → JSON deserialize

The main design choice: do you want a **password-based** key (user supplies a passphrase each time) or a **generated key** stored separately (e.g. in an env var or key file)?

## Encryption at rest

In [ ]:
#| eval: false
from cryptography.fernet import Fernet

key = Fernet.generate_key()    # bytes, save this somewhere safe
f = Fernet(key)

encrypted = f.encrypt(b"hello")  # bytes in → bytes out
f.decrypt(encrypted)             # b"hello"

encrypted

b'gAAAAABpryK1JUzcc9gKfdwm7qWkK6JxVKVeVBRA1U3Bi7YPr2V7LyEKJ5nN32-JtDsrSxfUW5d8mgpdHdR7HpbeJ9LVuQv_0w=='

Assume I use Fernet. Let's rethink the class to have this option to generate the key and store it first. I want it outside its state like a static method maybe

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good call — keeping the key outside instance state means the `Unknown` object itself is never holding the secret. Clean separation.

The shape would be:

- **`Unknown.generate_key()`** — `@staticmethod`, returns a Fernet key (bytes). User stores it however they like (env var, file, vault)
- **`save(path, key)`** — serialise `key_maps` + `vault` to JSON, encrypt with the provided key, write to file
- **`load(path, key)`** — read file, decrypt, deserialise, return a new `Unknown` instance (or populate `key_maps`/`vault`)

So the key is always **passed in** at save/load time, never stored on `self`. The user is responsible for keeping it safe.

Want to start by adding the `generate_key` staticmethod?

In [ ]:
#| eval: false
class CloakDF(object):
    def __init__(self, tables):
        self.tables = tables
        self.vault = {}
        self.key_maps = {}

    def encode(self, name, df):
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def _remap(self, df, col, mapping, id_fn):
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn()) # add a new key if not found to val (id_fn() return)
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        df = df.copy()
        
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def decode(self, name, df): pass

    @staticmethod
    def generate_key(path=None):
        from cryptography.fernet import Fernet
        key = Fernet.generate_key()
        if path:
            Path(path).write_bytes(key)
        return key
    
    @staticmethod
    def load_key(path=None, env_var=None):
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")


In [ ]:
#| eval: false
# key = CloakDF.generate_key()
# key.decode()

let's discuss where to include the fernet key load. I would argue at init. the user should have one set prior to using this 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

That's a reasonable argument — it ensures encryption is always available. But consider the trade-off:

**Key at `__init__`** — enforces that encryption is set up from the start. `save()`/`load()` can just use `self._key` internally. Clean API.

**Key at `save()`/`load()` only** — someone who just wants to `encode` a DataFrame in memory (no persistence) doesn't need to deal with keys at all. Lower barrier for simple use cases.

A middle ground: make it **optional** at init, but **required** for save/load. Something like:

```python
def __init__(self, tables, key=None):
```

If `key` is set, `save()` just works. If not, `save()` raises a clear error. That way casual usage stays simple, but the pipeline path is solid.

What's your preference?

In [ ]:
#| eval: false
from fastcore.basics import patch
import json

@patch
def save(self: CloakDF, path, key):
    """Encrypt and save key_maps + vault to file."""
    from cryptography.fernet import Fernet
    data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
    encrypted = Fernet(key).encrypt(data.encode())
    Path(path).write_bytes(encrypted)

@patch(cls_method=True)
def load(cls: CloakDF, path, key, tables):
    """Load encrypted mappings and return a new CloakDF instance."""
    from cryptography.fernet import Fernet
    encrypted = Path(path).read_bytes()
    data = json.loads(Fernet(key).decrypt(encrypted))
    unk = cls(tables)
    unk.key_maps, unk.vault = data['key_maps'], data['vault']
    return unk

In [ ]:
#| eval: false
bad_key = CloakDF.generate_key()
try:
    CloakDF.load(dest/"mappings.enc", bad_key, tables)
    assert False, "Should have raised"
except Exception as e:
    print(f"✓ Correctly rejected: {type(e).__name__}")

✓ Correctly rejected: InvalidToken


In [ ]:
#| eval: false
@patch
def decode(self: CloakDF, name, df):
    """Reverse pseudonymisation and masking for key columns and masked columns"""
    df = df.copy()
    tbl = self.tables[name]
    # Inverse key_maps for each id-group
    ids = {k: v for k, v in tbl.items() if k.startswith('id')}
    for group, col in ids.items():
        km = self.key_maps[group]
        inv_km = {v: k for k, v in km.items()}
        df[col] = df[col].map(inv_km)
    # Masked columns
    inv_vault = {v: k for k, v in self.vault.items()}
    for col in tbl.get('mask', []):
        df[col] = df[col].map(inv_vault)
    return df

In [ ]:
#| eval: false
import os

# 1. Generate key and store in env var
key = CloakDF.generate_key()
os.environ['UNKNOWN_KEY'] = key.decode()
loaded_key = CloakDF.load_key(env_var='UNKNOWN_KEY')
assert loaded_key == key
print("✓ Key round-tripped via env var")

# 2. Encode all tables
unk = CloakDF(tables)
originals, encoded = {}, {}
for name in tables:
    df = pd.read_csv(dest/f"{name}.csv", on_bad_lines='skip')
    for k, v in tables[name].items():
        if k.startswith('id'): df[v] = df[v].astype(str)
    originals[name] = df
    encoded[name] = unk.encode(name, originals[name])
    print(f"✓ Encoded {name}")

# 3. Save, load with env key, decode and compare
unk.save(dest/"mappings.enc", loaded_key)
unk2 = CloakDF.load(dest/"mappings.enc", loaded_key, tables)
for name in tables:
    decoded = unk2.decode(name, encoded[name])
    pd.testing.assert_frame_equal(decoded, originals[name])
    print(f"✓ Decoded {name} matches original")

✓ Key round-tripped via env var
✓ Encoded customers
✓ Encoded orders
✓ Encoded employees
✓ Decoded customers matches original
✓ Decoded orders matches original
✓ Decoded employees matches original


## Final class & round-trip test

In [ ]:
#| export
#| export
import uuid, json
from pathlib import Path
from cryptography.fernet import Fernet
import pandas as pd


class CloakDF:
    """Pseudonymise and mask DataFrames consistently across related tables."""
    def __init__(self, tables):
        self.tables,self.vault,self.key_maps = tables,{},{}

    def encode(self, name, df):
        """Pseudonymise keys and mask columns for table `name`, returning new DataFrame."""
        df = self._pseudonymise_keys(name, df)
        return self._mask_columns(name, df)

    def decode(self, name, df):
        """Reverse `encode` — restore original keys and masked values."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            inv = {v: k for k, v in self.key_maps[group].items()}
            df[col] = df[col].map(inv)
        inv_vault = {v: k for k, v in self.vault.items()}
        for col in tbl.get('mask', []):
            df[col] = df[col].map(inv_vault)
        return df

    def _remap(self, df, col, mapping, id_fn):
        """Replace `col` values via `mapping`, adding new entries with `id_fn`."""
        for val in df[col].dropna().unique():
            mapping.setdefault(str(val), id_fn())
        df[col] = df[col].astype(str).map(mapping)
        return df

    def _pseudonymise_keys(self, name, df):
        """Replace id columns with consistent UUIDs via `key_maps`."""
        df = df.copy()
        tbl = self.tables[name]
        ids = {k: v for k, v in tbl.items() if k.startswith('id')}
        for group, col in ids.items():
            assert df[col].notna().all(), f"Null found in {col} for table {name}"
            assert pd.api.types.is_string_dtype(df[col]), f"Column {col} must be string type, got {df[col].dtype}"
            km = self.key_maps.setdefault(group, {})
            df = self._remap(df, col, km, lambda: str(uuid.uuid4()))
        return df

    def _mask_columns(self, name, df):
        """Replace mask columns with truncated hex tokens via shared `vault`."""
        tbl = self.tables[name]
        for col in tbl.get('mask', []):
            df = self._remap(df, col, self.vault, lambda: uuid.uuid4().hex[:12])
        return df

    def save(self, path, key):
        """Encrypt and save `key_maps` + `vault` to `path` using Fernet `key`."""
        data = json.dumps({'key_maps': self.key_maps, 'vault': self.vault})
        Path(path).write_bytes(Fernet(key).encrypt(data.encode()))

    @classmethod
    def load(cls, path, key, tables):
        """Load encrypted mappings from `path` and return a new `CloakDF` instance."""
        data = json.loads(Fernet(key).decrypt(Path(path).read_bytes()))
        unk = cls(tables)
        unk.key_maps, unk.vault = data['key_maps'], data['vault']
        return unk

    @staticmethod
    def generate_key(path=None):
        """Generate a Fernet key, optionally saving to `path`."""
        key = Fernet.generate_key()
        if path: Path(path).write_bytes(key)
        return key

    @staticmethod
    def load_key(path=None, env_var=None):
        """Load a Fernet key from `path` or `env_var`."""
        if path: return Path(path).read_bytes()
        if env_var:
            import os
            return os.environ[env_var].encode()
        raise ValueError("Provide either path or env_var")

In [ ]:
#| eval: false
%load_ext autoreload
%autoreload 2

In [ ]:
#| eval: false
from cloakdf.core import CloakDF

ModuleNotFoundError: No module named 'cloakdf'

Let's see if we can commit the latest changes 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's start by checking what's changed:

```bash
!git status
```

Run that and we'll see what needs staging. You'll probably also want to run `!nbdev_export` first if you haven't already, to make sure the module file is in sync with the notebook.

In [ ]:
#| eval: false
!git status

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   00_core.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


ok i need to publish this package to pypi

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

A few things to sort out before publishing. First, let's check your package settings:

```bash
!cat settings.ini
```

This will show us the metadata (version, description, author, etc.) that PyPI needs. Run that and we'll review what needs updating.

In [ ]:
! ls ../

LICENSE      README.md	data  nbs	      unknown
MANIFEST.in  _proc	dist  pyproject.toml  unknown.egg-info


what is the setting file you are looking for?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Ah sorry — I was looking for `settings.ini` (the older nbdev convention), but your project has `pyproject.toml` instead. That's the one we need. Let's look at it:

```bash
!cat ../pyproject.toml
```

In [ ]:
!cat ../pyproject.toml

[build-system]
requires = ["setuptools>=64"]
build-backend = "setuptools.build_meta"

[project]
name = "unknown"
dynamic = ["version"]
description = "Pseudonymisation/anonymisation engine with encrypted mapping storage"
readme = "README.md"
requires-python = ">=3.10"
license = {text = "Apache-2.0"}
authors = [{name = "frangs", email = "FGiordano-Silva@lambeth.gov.uk"}]
keywords = ['nbdev']
classifiers = [
    "Programming Language :: Python :: 3",
    "Programming Language :: Python :: 3 :: Only",
]
dependencies = ["cryptography", "pandas"]

[project.urls]
Repository = "https://github.com/frangs/unknown"
Documentation = "https://frangs.github.io/unknown/"

[project.entry-points.nbdev]
unknown = "unknown._modidx:d"

[tool.setuptools.dynamic]
version = {attr = "unknown.__version__"}

[tool.setuptools.packages.find]
include = ["unknown"]

[tool.nbdev]


In [ ]:
!pip index versions unknown

ERROR: No matching distribution found for unknown


ok what are we missing here. dependencis and description added

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The metadata looks good. Two things to check before building:

1. **Version** — your `pyproject.toml` reads version from `unknown.__version__`. Let's check it's set:

```bash
!cat ../unknown/__init__.py
```

2. **Package name** — `"unknown"` is extremely generic. PyPI might reject it or it could clash later. Have you considered something more specific like `unknown-anon` or `pseudonymiser`?

Run that `cat` and we'll see where the version stands.

In [ ]:
!cat ../unknown/__init__.py

__version__ = "0.0.1"


ok what next steps i belive ive setup my pypi token also

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good — version is set and token is ready. The steps are:

1. **Build** the distribution:
   ```bash
   !cd .. && python -m build
   ```

2. **Upload** to PyPI:
   ```bash
   !cd .. && twine upload dist/*
   ```

You may need to install `build` and `twine` first if they're not already available:
```bash
!pip install build twine
```

Start with the install and build step, and we'll see if it compiles cleanly.

i assume the built and all of the above are run in the project root folder where pyproject is?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes, exactly — `python -m build` and `twine upload` both expect to be run from the directory containing `pyproject.toml`. That's why the commands I gave used `cd ..` — since your notebook is in the `nbs/` subfolder.

So from a terminal it would just be:

```bash
cd /path/to/unknown   # where pyproject.toml lives
python -m build
twine upload dist/*
```

Or from the notebook, keep using `!cd .. && ...` to run from the project root.

In [ ]:
!cd .. && twine upload dist/* --v

INFO     Using configuration from /app/data/.pypirc                             
Uploading distributions to https://upload.pypi.org/legacy/
INFO     dist/unknown-0.0.1-py3-none-any.whl (10.2 KB)                          
INFO     dist/unknown-0.0.1.tar.gz (9.8 KB)                                     
INFO     username set by command options                                        
INFO     password set from config file                                          
INFO     username: __token__                                                    
INFO     password: <hidden>                                                     
Uploading unknown-0.0.1-py3-none-any.whl
  0% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/19.4 kB • --:-- • ?

100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 kB • 00:00 • 15.3 MB/s

100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 kB • 00:00 • 15.3 MB/s
INFO     Response from https://upload.pypi.org/legacy/:                         
         403 Forbidden                                                          
INFO     <html>                                                                 
          <head>                                                                
           <title>403 The user 'giordafrancis' isn't allowed to upload to       
         project 'UNKNOWN'. See https://pypi.org/help/#project-name for more    
         information.</title>                                                   
          </head>                                                               
          <body>                                                                
           <h1>403 The user 'giordafrancis' isn't allowed to upload to project  
         'UNKNOWN'. See https://pypi.org/help/#project-name for more            
         information.</h1>    

we are changing th ename to cloakdf

lets change everything in core for this. First id what needs to be changed

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good name choice! Here's what needs changing in the core notebook:

**Exported code (the final class cell):**
- `class Unknown:` → new class name (e.g. `CloakDF`?)
- Docstrings that reference "Unknown"

**Test/exploration cells throughout:**
- `unk = Unknown(tables)` — multiple cells
- `Unknown.generate_key()`
- `Unknown.load(...)`, `Unknown.load_key(...)`
- `from unknown.core import Unknown`

**Outside core (but we can do later):**
- `pyproject.toml` — package name, module references
- `unknown/` folder → `cloakdf/`
- `README.md`, `index.ipynb`

First question — what do you want the **class name** to be? `CloakDF`? `Cloak`? Something else?

lets move to index. also change refs in note cells

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let me check what's in the index notebook and find the note cells that need updating.

<details class='tool-usage-details'>
<summary>view(path='../index.ipynb')</summary>

```json
{
  "id": "toolu_01J3oSAP43AHhj6bQUTyRyVW",
  "call": {
    "function": "view",
    "arguments": {
      "path": "../index.ipynb"
    }
  },
  "result": "'Error: File not found: /app/data/wk/unknown/unknown/index.ipynb'"
}
```

</details>



<details class='tool-usage-details'>
<summary>find_msgs(msg_type='note', re_pattern='[Uu]nknown')</summary>

```json
{
  "id": "toolu_01HkPu3y7brpcyG3F3TuSnqP",
  "call": {
    "function": "find_msgs",
    "arguments": {
      "msg_type": "note",
      "re_pattern": "[Uu]nknown"
    }
  },
  "result": "[]"
}
```

</details>



<details class='tool-usage-details'>
<summary>view(path='..')</summary>

```json
{
  "id": "toolu_01CbELB1A2koHVjSTZ3pn39f",
  "call": {
    "function": "view",
    "arguments": {
      "path": ".."
    }
  },
  "result": "<TRUNCATED>\u2026 contents of /app/data/wk/unknown/unknown:\\n/app/data/wk/unknown/unknown/dist \ud83d\udcc1\\n/app/data/wk/unknown/unknown/unknown.egg-info \ud83d\udcc1\\n/app/data/wk/unknown/unknown/nbs \ud83d\udcc1\\n/app/data/wk/unknown/unknown/LICENSE (11.1k)\\n/app/data/wk/unknown/unknown/README.md (5.9k)\\n/app/data/wk/unknown/unknown/pyproject.toml (0.8k)\\n/app/data/wk/unknown/unknown/unknown \ud83d\udcc1\\n/app/data/wk/unknown/unknown/MANIFEST.in (0.1k)\\n/app/data/wk/unknown/unknown/data \ud83d\udcc1\\n/app/data/wk/unknown/unknown/dist/unknown-0.0.1-py3-none-any.whl (10.2k)\\n/app/data/wk/unknown/unknown/dist/unknown-0.0.1.tar.gz (9.8k)\\n/app/data/wk/unknown/unknown/unknown.egg-info/top_level.txt (0.0k)\\n/app/data/wk/unknown/unknown/unknown.egg-info/PKG-INFO (6.5k)\\n/app/data/wk/unknown/unknown/unknown.egg-info/entry_points.txt (0.0k)\\n/app/data/wk/unknown/unknown/unknown.egg-info/SOURCES.txt (0.3k)\\n/app/data/wk/unknown/unknown/unknown.egg-info/dependency_links.txt (0.0k)\\n/app/data/wk/unknown/unknown/unknown.egg-info/requires.txt (0.0\u2026</TRUNCATED>"
}
```

</details>



<details class='tool-usage-details'>
<summary>view(path='../nbs/index.ipynb')</summary>

```json
{
  "id": "toolu_014gpbvEH6htn29H5QLTPHpg",
  "call": {
    "function": "view",
    "arguments": {
      "path": "../nbs/index.ipynb"
    }
  },
  "result": "<TRUNCATED>\u2026s\": [\\n  {\\n   \"cell_type\": \"markdown\",\\n   \"id\": \"a60b6e78\",\\n   \"metadata\": {},\\n   \"source\": [\\n    \"# unknown\\\\n\",\\n    \"\\\\n\",\\n    \"> A pseudonymisation/anonymisation package for consistent cross-table key and column masking, with encrypted storage of mappings.\\\\n\",\\n    \"\\\\n\",\\n    \"When working with related datasets that share identifiers (e.g. `customerID` appearing in both a customers and orders table), anonymisation must be **consistent** \u2014 the same original ID should map to the same pseudonym everywhere. `unknown` handles this automatically via shared key groups, while separately masking sensitive columns (names, addresses, phone numbers) with opaque hex tokens. All mappings are stored encrypted at rest using Fernet symmetric encryption.\"\\n   ]\\n  },\\n  {\\n   \"cell_type\": \"code\",\\n   \"execution_count\": null,\\n   \"id\": \"58d073c7\",\\n   \"metadata\": {},\\n   \"outputs\": [],\\n   \"source\": [\\n    \"#| hide\\\\n\",\\n    \"from unknown.core import *\"\\n   ]\\n  },\\n  {\\n \u2026</TRUNCATED>"
}
```

</details>



🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

I can see the `index.ipynb` has lots of `Unknown` and `unknown` references. Here's what needs changing:

**In `index.ipynb`:**
- Title: `# unknown` → `# cloakdf`
- Prose: `` `unknown` handles this`` → `` `cloakdf` handles this`` (×2)
- Import: `from unknown.core import *` → `from cloakdf.core import *`
- Code: `Unknown(tables)` → `CloakDF(tables)` (×3 instances)
- Code: `Unknown.generate_key()` → `CloakDF.generate_key()`
- Code: `Unknown.load_key(...)` → `CloakDF.load_key(...)` (×2)
- Code: `Unknown.load(...)` → `CloakDF.load(...)`
- Env var: `UNKNOWN_KEY` → `CLOAKDF_KEY` (×3)

No note cells in the current dialog reference `Unknown` — the class was already renamed to `CloakDF` here.

Shall I go ahead and make these changes to `index.ipynb`?

proceeed please

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>
<summary>file_strs_replace(path='../nbs/index.ipynb', old_strs="<…>…n', '\`unknown\` handles this'…</…>", new_strs="<…>…f', '\`cloakdf\` handles this'…</…>")</summary>

```json
{
  "id": "toolu_016G978ugza5NUPj2YNadUeY",
  "call": {
    "function": "file_strs_replace",
    "arguments": {
      "path": "../nbs/index.ipynb",
      "old_strs": "['# unknown', '`unknown` handles this', 'from unknown.core import', 'Unknown(tables)', 'Unknown.generate_key()', 'Unknown.load_key(', 'Unknown.load(', 'UNKNOWN_KEY']",
      "new_strs": "['# cloakdf', '`cloakdf` handles this', 'from cloakdf.core import', 'CloakDF(tables)', 'CloakDF.generate_key()', 'CloakDF.load_key(', 'CloakDF.load(', 'CLOAKDF_KEY']"
    }
  },
  "result": "<TRUNCATED>\u20267,3 @@\\n    \"source\": [\\n-    \"# unknown\\\\n\",\\n+    \"# cloakdf\\\\n\",\\n     \"\\\\n\",\\n@@ -11,3 +11,3 @@\\n     \"\\\\n\",\\n-    \"When working with related datasets that share identifiers (e.g. `customerID` appearing in both a customers and orders table), anonymisation must be **consistent** \u2014 the same original ID should map to the same pseudonym everywhere. `unknown` handles this automatically via shared key groups, while separately masking sensitive columns (names, addresses, phone numbers) with opaque hex tokens. All mappings are stored encrypted at rest using Fernet symmetric encryption.\"\\n+    \"When working with related datasets that share identifiers (e.g. `customerID` appearing in both a customers and orders table), anonymisation must be **consistent** \u2014 the same original ID should map to the same pseudonym everywhere. `cloakdf` handles this automatically via shared key groups, while separately masking sensitive columns (names, addresses, phone numbers) with opaque hex\u2026</TRUNCATED>"
}
```

</details>

Now let me check if there are any remaining `unknown` references (lowercase, as package name):

<details class='tool-usage-details'>
<summary>rg(argstr='-i "unknown" ../nbs/index.ipynb')</summary>

```json
{
  "id": "toolu_01E5Gr44gQnVLjE2LrVahqV7",
  "call": {
    "function": "rg",
    "arguments": {
      "argstr": "-i \"unknown\" ../nbs/index.ipynb"
    }
  },
  "result": "'35:    \"We\\'ll demonstrate `unknown` using the [Northwind dataset](https://github.com/neo4j-contrib/northwind-neo4j). A classic sample database with customers, orders, and employees tables that share keys across them.\\\\n\",\\n'"
}
```

</details>

One more reference — let me fix it:

<details class='tool-usage-details'>
<summary>file_str_replace(path='../nbs/index.ipynb', old_str='demonstrate \`unknown\` using', new_str='demonstrate \`cloakdf\` using')</summary>

```json
{
  "id": "toolu_018hoeFpVjhigRLKKQxRdkaQ",
  "call": {
    "function": "file_str_replace",
    "arguments": {
      "path": "../nbs/index.ipynb",
      "old_str": "demonstrate `unknown` using",
      "new_str": "demonstrate `cloakdf` using"
    }
  },
  "result": "'@@ -34,3 +34,3 @@\\n     \"\\\\n\",\\n-    \"We\\'ll demonstrate `unknown` using the [Northwind dataset](https://github.com/neo4j-contrib/northwind-neo4j). A classic sample database with customers, orders, and employees tables that share keys across them.\\\\n\",\\n+    \"We\\'ll demonstrate `cloakdf` using the [Northwind dataset](https://github.com/neo4j-contrib/northwind-neo4j). A classic sample database with customers, orders, and employees tables that share keys across them.\\\\n\",\\n     \"\\\\n\",'"
}
```

</details>

All done in `index.ipynb`. Here's a summary of changes:

- **Title**: `# unknown` → `# cloakdf`
- **Prose**: 3 references to `` `unknown` `` → `` `cloakdf` ``
- **Import**: `from unknown.core` → `from cloakdf.core`
- **Class refs**: `Unknown(...)` → `CloakDF(...)` (×5 occurrences)
- **Env var**: `UNKNOWN_KEY` → `CLOAKDF_KEY` (×3)

Still to do outside this notebook: `pyproject.toml`, the `unknown/` folder rename, and `README.md`. Want to tackle those next?

In [ ]:
!pwd

/app/data/wk/cloakdf/nbs


In [ ]:
!ls

00_core.ipynb  _quarto.yml  index.ipynb  nbdev.yml  styles.css


In [ ]:
!ls ../

LICENSE      README.md	cloakdf  dist  pyproject.toml
MANIFEST.in  _proc	data	 nbs


ive rebamed a few folder lets clean the pyproject file

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let me see the current state:

<details class='tool-usage-details'>
<summary>view(path='../pyproject.toml', nums=True)</summary>

```json
{
  "id": "toolu_01XQZfSdEnRvHhTLkTTTRKM7",
  "call": {
    "function": "view",
    "arguments": {
      "path": "../pyproject.toml",
      "nums": "True"
    }
  },
  "result": "<TRUNCATED>\u2026[build-system]\\n     2 \u2502 requires = [\"setuptools>=64\"]\\n     3 \u2502 build-backend = \"setuptools.build_meta\"\\n     4 \u2502 \\n     5 \u2502 [project]\\n     6 \u2502 name = \"unknown\"\\n     7 \u2502 dynamic = [\"version\"]\\n     8 \u2502 description = \"Pseudonymisation/anonymisation engine with encrypted mapping storage\"\\n     9 \u2502 readme = \"README.md\"\\n    10 \u2502 requires-python = \">=3.10\"\\n    11 \u2502 license = {text = \"Apache-2.0\"}\\n    12 \u2502 authors = [{name = \"frangs\", email = \"FGiordano-Silva@lambeth.gov.uk\"}]\\n    13 \u2502 keywords = [\\'nbdev\\']\\n    14 \u2502 classifiers = [\\n    15 \u2502     \"Programming Language :: Python :: 3\",\\n    16 \u2502     \"Programming Language :: Python :: 3 :: Only\",\\n    17 \u2502 ]\\n    18 \u2502 dependencies = [\"cryptography\", \"pandas\"]\\n    19 \u2502 \\n    20 \u2502 [project.urls]\\n    21 \u2502 Repository = \"https://github.com/frangs/unknown\"\\n    22 \u2502 Documentation = \"https://frangs.github.io/unknown/\"\\n    23 \u2502 \\n    24 \u2502 [project.entry-points.nbdev]\\n    25 \u2502 unknown = \"unknown._modidx:d\"\\n    26 \u2502 \\n  \u2026</TRUNCATED>"
}
```

</details>

Here's what needs changing — lines 6, 21, 22, 25, 28, and 31 all reference `unknown`:

<details class='tool-usage-details'>
<summary>file_strs_replace(path='../pyproject.toml', old_strs='<…>…unknown"\', \'github.com/frangs/…</…>', new_strs='<…>…cloakdf"\', \'github.com/frangs/…</…>')</summary>

```json
{
  "id": "toolu_01Qb998th4KQduZjtvM2qZUX",
  "call": {
    "function": "file_strs_replace",
    "arguments": {
      "path": "../pyproject.toml",
      "old_strs": "['name = \"unknown\"', 'github.com/frangs/unknown', 'frangs.github.io/unknown/', 'unknown = \"unknown._modidx:d\"', 'attr = \"unknown.__version__\"', 'include = [\"unknown\"]']",
      "new_strs": "['name = \"cloakdf\"', 'github.com/frangs/cloakdf', 'frangs.github.io/cloakdf/', 'cloakdf = \"cloakdf._modidx:d\"', 'attr = \"cloakdf.__version__\"', 'include = [\"cloakdf\"]']"
    }
  },
  "result": "'@@ -5,3 +5,3 @@\\n [project]\\n-name = \"unknown\"\\n+name = \"cloakdf\"\\n dynamic = [\"version\"]\\n@@ -20,13 +20,13 @@\\n [project.urls]\\n-Repository = \"https://github.com/frangs/unknown\"\\n-Documentation = \"https://frangs.github.io/unknown/\"\\n+Repository = \"https://github.com/frangs/cloakdf\"\\n+Documentation = \"https://frangs.github.io/cloakdf/\"\\n \\n [project.entry-points.nbdev]\\n-unknown = \"unknown._modidx:d\"\\n+cloakdf = \"cloakdf._modidx:d\"\\n \\n [tool.setuptools.dynamic]\\n-version = {attr = \"unknown.__version__\"}\\n+version = {attr = \"cloakdf.__version__\"}\\n \\n [tool.setuptools.packages.find]\\n-include = [\"unknown\"]\\n+include = [\"cloakdf\"]\\n '"
}
```

</details>

All six references updated. Want to check `README.md` next, or go straight to rebuilding?

In [ ]:
!nbdev-readme

pandoc -o README.md
  to: >-
    commonmark+autolink_bare_uris+emoji+footnotes+gfm_auto_identifiers+pipe_tables+strikeout+task_lists+tex_math_dollars
  output-file: index.html
  standalone: true
  default-image-extension: png
  variables: {}
  
metadata
  description: A pseudonymisation/anonymisation package for consistent cross-table key and column masking, with encrypted storage of mappings.
  title: cloakdf
  


Output created: _docs/README.md



ive rebuild the readme what is left before reinstall and republish?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

A few things to verify:

1. **`nbdev.yml`** — likely still references `unknown` as the lib name
2. **`nbdev_export`** — regenerate the module files into `cloakdf/`
3. **Clean old dist** — `dist/` has the old `unknown-0.0.1` files
4. **Bump version?** — 0.0.1 was already uploaded (even though it failed), you may want 0.0.2

Let's start by checking `nbdev.yml`:

```bash
!cat nbdev.yml
```

In [ ]:
!cat nbdev.yml

project:
  output-dir: _docs

website:
  title: "cloakdf"
  site-url: "https://frangs.github.io/cloakdf"
  description: ""
  repo-branch: main
  repo-url: "https://github.com/frangs/cloakdf"


lets start here please amend this

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->



<details class='tool-usage-details'>
<summary>file_strs_replace(path='nbdev.yml', old_strs='<…>…unknown"\', \'frangs.github.io/u…</…>', new_strs='<…>…cloakdf"\', \'frangs.github.io/c…</…>')</summary>

```json
{
  "id": "toolu_01KLoSNyyG8MhaNotvRGPGkC",
  "call": {
    "function": "file_strs_replace",
    "arguments": {
      "path": "nbdev.yml",
      "old_strs": "['title: \"unknown\"', 'frangs.github.io/unknown', 'github.com/frangs/unknown']",
      "new_strs": "['title: \"cloakdf\"', 'frangs.github.io/cloakdf', 'github.com/frangs/cloakdf']"
    }
  },
  "result": "'@@ -4,6 +4,6 @@\\n website:\\n-  title: \"unknown\"\\n-  site-url: \"https://frangs.github.io/unknown\"\\n+  title: \"cloakdf\"\\n+  site-url: \"https://frangs.github.io/cloakdf\"\\n   description: \"\"\\n   repo-branch: main\\n-  repo-url: \"https://github.com/frangs/unknown\"\\n+  repo-url: \"https://github.com/frangs/cloakdf\"'"
}
```

</details>

Done. Next up — clean the old dist and run `nbdev_export`:

```bash
!rm -rf ../dist && nbdev-export
```

In [ ]:
!rm -rf ../dist && nbdev-export

ok next

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Check the version and bump it, then rebuild:

```bash
!cat ../cloakdf/__init__.py
```

If it's still `0.0.1`, bump it to `0.0.2` before building.

In [ ]:
!cat ../cloakdf/__init__.py

__version__ = "0.0.2"


ie bumped the version next ??

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good — 0.0.2 is set. Now build and upload:

```bash
!cd .. && python -m build
```

Once that completes cleanly, we'll upload with twine.

shoudl i commit all to remote

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Yes — good idea to commit before publishing. That way your repo matches what's on PyPI. The sequence:

```bash
!cd .. && git add -A && git status
```

Review what's staged, then commit and push, then build and upload.

In [ ]:
!cd .. && git add -A && git status